In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [2]:

url = "https://www.sharesansar.com/today-share-price"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

In [3]:
soup = BeautifulSoup(response.content, "html.parser")
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, user-scalable=no, initial-scale = 1.0, minimum-scale=1.0, maximum-scale = 1.0" name="viewport"/>
  <meta content=" sharesansar, sharesansar.com, share market, best site, nepali bazar, news, nepal, economics, politics, entertainment, business, politics, businessman, online news, real state, tax, banking, corporate, telecom,  nepse, ipo, agm, bond, Oil, Gold, profit, shareholder, NEPSE, pravhu, ILFC, Stock, exchange, citizen, nepal., loan, loss, provision,  arun, valley, kabeli,  Economy, Growth, Rate, Eid, Public, Holiday,  Mastercard, Standard, Chartered, Credit, Card, Visa, Warren, Buffett, Value, Investing,Nepal Stock Exchange (NEPSE) Live Trading Data, Floorsheet, Live Indices, Top Gainers, Top Losers, nepse, shareapp, limted, iporesult, bank , share, bank, company, promoter, bittya santha" name="keywords">
   <title>
    Today Share Price - || ShareSansar ||
   </title>
   <!--[i

In [4]:
# ============================================================
# FIND TABLE
# ============================================================

table = soup.find("table", id="headFixed")

if table is None:
    raise ValueError("Table with id='headFixed' not found")

print("Table found successfully!")

Table found successfully!


In [5]:
# ============================================================
# GET TABLE HEADERS
# ============================================================

headers = []

for th in table.find_all("th"):
    header = th.get_text(" ", strip=True)
    headers.append(header)

print("Headers:")
print(headers)

print("\nNumber of headers:", len(headers))


# ============================================================
# EXTRACT ALL ROW DATA
# ============================================================

data = []

tbody = table.find("tbody")

if tbody is None:
    raise ValueError("Table body not found")


for row in tbody.find_all("tr"):

    cells = row.find_all("td")

    # Skip empty rows
    if not cells:
        continue

    row_data = []

    for cell in cells:
        value = cell.get_text(" ", strip=True)
        row_data.append(value)

    # Only keep rows having the same number of cells
    # as the table headers
    if len(row_data) == len(headers):
        data.append(row_data)

    else:
        print(
            f"Skipping row: expected {len(headers)} cells, "
            f"found {len(row_data)}"
        )


# ============================================================
# CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    data,
    columns=headers
)


# ============================================================
# ADD MARKET DATE
# ============================================================

date_element = soup.select_one("p span.text-org")

if date_element is not None:

    market_date = date_element.get_text(
        strip=True
    )

    df["Market Date"] = market_date

else:

    df["Market Date"] = None


# ============================================================
# CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


# ============================================================
# CLEAN NUMERIC COLUMNS
# ============================================================

# Columns that should contain numbers
numeric_columns = [
    "S.No",
    "Conf.",
    "Open",
    "High",
    "Low",
    "Close",
    "LTP",
    "Close - LTP",
    "Close - LTP %",
    "VWAP",
    "Vol",
    "Prev. Close",
    "Turnover",
    "Trans.",
    "Diff",
    "Range",
    "Diff %",
    "Range %",
    "VWAP %",
    "120 Days",
    "180 Days",
    "52 Weeks High",
    "52 Weeks Low"
]


for col in numeric_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ============================================================
# DISPLAY RESULT
# ============================================================

print("\n======================================")
print("FINAL DATAFRAME")
print("======================================")

print(df)


print("\n======================================")
print("SHAPE")
print("======================================")

print(df.shape)


print("\n======================================")
print("COLUMNS")
print("======================================")

print(df.columns.tolist())


print("\n======================================")
print("DATA TYPES")
print("======================================")

print(df.dtypes)

Headers:
['S.No', 'Symbol', 'Conf.', 'Open', 'High', 'Low', 'Close', 'LTP', 'Close - LTP', 'Close - LTP %', 'VWAP', 'Vol', 'Prev. Close', 'Turnover', 'Trans.', 'Diff', 'Range', 'Diff %', 'Range %', 'VWAP %', '120 Days', '180 Days', '52 Weeks High', '52 Weeks Low']

Number of headers: 24

FINAL DATAFRAME
     S.No   Symbol  Conf.    Open    High     Low   Close     LTP  \
0       1   ACLBSL  25.91   915.0   915.0   901.2   901.2   901.2   
1       2     ADBL  30.92   302.0   302.0   295.5   300.0   300.0   
2       3  ADBLB87  22.09  1015.0  1015.0  1012.0  1012.0  1012.0   
3       4      AHL  29.65   415.0   415.0   403.1   408.0   408.0   
4       5     AHPC  35.63   263.6   264.0   260.0   260.0   260.0   
..    ...      ...    ...     ...     ...     ...     ...     ...   
342   343     USLB  35.68  1116.0  1118.0  1096.0  1105.0  1105.0   
343   344     VLBS  40.04   658.9   676.0   653.1   663.0   663.0   
344   345    VLUCL  36.87   407.0   407.0   396.0   397.0   397.0   
345  

In [6]:
# Sector-wise script lists
Bank = ["NABIL", "NIMB", "SCB", "HBL", "SBI", "EBL", "NICA", "MBL", "LSL", "KBL",
        "SBL", "SANIMA", "NMB", "PRVU", "GBIME", "CZBIL", "PCBL", "ADBL", "NBL"]

manufacturing = ["BNL", "NLO", "BNT", "UNL", "HDL", "SHIVM", "GCIL", "SONA",
                 "SARBTM", "OMPL", "SAGAR", "SAIL", "SYPNL", "RSML", "PCIL", "SOPL", "ECL"]

hotel_and_tourism = ["SHL", "TRH", "OHL", "CGH", "KDL", "CITY", "BANDIPUR", "HFIN"]

other = ["NTC", "NRIC", "NRM", "MKCL", "NWCL", "HRL", "PURE", "TTL"]

hydropower = [
    "NHPC", "BPCL", "CHCL", "AHPC", "SHPC", "RIDI", "BARUN", "API",
    "NGPL", "KKHC", "DHPL", "AKPL", "SPDL", "UMHL", "CHL", "HPPL",
    "NHDL", "RADHI", "PMHPL", "KPCL", "AKJCL", "JOSHI", "UPPER", "GHL",
    "UPCL", "MHNL", "PPCL", "HURJA", "UNHPL", "RHPL", "SJCL", "HDHPC",
    "LEC", "SSHL", "MEN", "UMRH", "GLH", "SHEL", "RURU", "MKJC",
    "SAHAS", "TPC", "SPC", "NYADI", "MBJC", "BNHC", "GVL", "BHL",
    "RFPL", "DORDI", "BHDC", "HHL", "UHEWA", "SGHC", "MHL", "USHEC",
    "RHGCL", "SPHL", "PPL", "SIKLES", "EHPL", "PHCL", "BHPL", "SMHL",
    "SPL", "SMH", "MKHC", "AHL", "TAMOR", "MHCL", "SMJC", "MAKAR",
    "MKHL", "DOLTI", "BEDC", "MCHL", "IHL", "MEL", "RAWA", "USHL",
    "TSHL", "KBSH", "MEHL", "ULHC", "MANDU", "BGWT", "MSHL", "MMKJL",
    "TVCL", "VLUCL", "CKHL", "SANVI", "BHCL", "HIMSTAR", "MABEL", "DHEL",
    "BUNGAL", "SOHL", "BJHL", "SKHL", "RLEL", "SKHEL", "SIPD", "KHPL",
    "APHL", "YMHL", "TPKHL", "SNORL", "SGHL", "KAHL", "MEPDL"
]

trading = ["STC", "BBC"]

non_life_insurance = [
    "NICL", "RBCL", "HEI", "UAIL", "SPIL", "NIL", "PRIN",
    "SALICO", "IGI", "SICL", "NLG", "SGIC", "NMIC"
]

development_bank = [
    "NABBC", "EDBL", "LBBL", "MDB", "MLBL", "GBBL", "JBBL", "CORBL",
    "KSBBL", "SADBL", "SHINE", "MNBBL", "SINDU", "GRDBL", "SAPDBL", "SABBL"
]

finance = [
    "NFS", "GUFL", "BFC", "GFCL", "SIFC", "CFCL", "JFL",
    "GMFIL", "ICFC", "PROFL", "MPFL", "MFIL", "RLFL"
]

microfinance = [
    "NUBL", "CBBL", "DDBL", "SWBBL", "NMLBBL", "FMDBL", "SLBBL", "SKBBL",
    "GBLBS", "KMCDB", "MLBBL", "LLBS", "VLBS", "HLBSL", "MATRI", "JSLBB",
    "NMBMF", "GILB", "SWMF", "MERO", "NMFBS", "RSDC", "FOWAD", "SMATA",
    "MSLB", "SMB", "USLB", "WNLB", "NADEP", "ACLBSL", "SLBSL", "ALBSL",
    "GMFBS", "GLBSL", "SMFBS", "ILBS", "NICLBSL", "SMPDA", "MLBSL", "JBLB",
    "MLBS", "NESDO", "ULBSL", "CYCL", "AVYAN", "DLBS", "SHLB", "UNLB",
    "ANLB", "SWASTIK"
]

life_insurance = [
    "NLICL", "NLIC", "LICN", "ALICL", "HLI", "SJLIC", "PMLI",
    "SRLI", "ILI", "RNLI", "SNLI", "CLI", "GMLI", "CREST"
]

investment = [
    "CIT", "HIDCL", "NRN", "NIFRA", "CHDC", "ENL", "HATHY"
]
mutual_fund = [
    "SEF", "NBF2", "SIGS2", "NICBF", "NMB50", "SFMF", "LUK", "SLCF",
    "KEF", "SBCF", "PSF", "NIBSF2", "NICSF", "RMF1", "MMF1", "NBF3",
    "NICFC", "KDBY", "GIBF1", "NSIF2", "NIBLGF", "SAGF", "SFEF", "PRSF",
    "RMF2", "SIGS3", "C30MF", "LVF2", "H8020", "NICGF2", "KSY", "NIBLSTF",
    "MNMF1", "GSY", "NMBHF2", "MBLEF", "RSY", "GBIMESY2", "HLICF", "RBBF40",
    "CSY", "NSY", "SEF2", "SAEF2", "LSH12", "RSY2"
]

preference_share = [
    "NABILPNP",
    "KSBBLPNP",
    "SBLPNP",
    "NMBPNP",
    "SANIMAPNP",
    "MBLPNP"
]

# Create sector mapping
sector_mapping = {}

for script in Bank:
    sector_mapping[script] = "Banking"

for script in manufacturing:
    sector_mapping[script] = "Manufacturing"

for script in hotel_and_tourism:
    sector_mapping[script] = "Hotel and Tourism"

for script in other:
    sector_mapping[script] = "Other"

for script in hydropower:
    sector_mapping[script] = "Hydropower"

for script in trading:
    sector_mapping[script] = "Trading"

for script in non_life_insurance:
    sector_mapping[script] = "Non-Life Insurance"

for script in development_bank:
    sector_mapping[script] = "Development Bank"

for script in finance:
    sector_mapping[script] = "Finance"

for script in microfinance:
    sector_mapping[script] = "Microfinance"

for script in life_insurance:
    sector_mapping[script] = "Life Insurance"

for script in investment:
    sector_mapping[script] = "Investment"

for script in mutual_fund:
    sector_mapping[script] = "Mutual Fund"

for script in preference_share:
    sector_mapping[script] = "Preference Share"

# Create sector column
df["sector"] = df["Symbol"].map(sector_mapping)

In [7]:
from pathlib import Path
from datetime import date

# Go from src/scraper -> project root
project_root = Path.cwd().parents[1]

# data/raw
raw_folder = project_root / "data" / "raw"/"stock_data"

# Create folder if it doesn't exist
raw_folder.mkdir(parents=True, exist_ok=True)

# Filename
today_date = date.today().strftime("%Y-%m-%d")
file_path = raw_folder / f"{today_date}-stock_data.csv"

# Save
df.to_csv(file_path, index=False)

print(f"Saved successfully: {file_path}")

Saved successfully: e:\nepse-market-report\data\raw\stock_data\2026-08-18-stock_data.csv


In [8]:
# for combining all the data in one file, we can use the following code:
# ============================================================
# 1. COPY SCRAPED DATA
# ============================================================

df2 = df.copy()


# ============================================================
# 2. ADD TODAY'S DATE
# ============================================================

today = date.today()

df2["Date"] = today


# ============================================================
# 3. GO FROM src/scraper -> PROJECT ROOT
# ============================================================

project_root = Path.cwd().parents[1]


# ============================================================
# 4. DATA/RAW/HISTORIC_STOCK_DATA FOLDER
# ============================================================

raw_folder = (
    project_root
    / "data"
    / "raw"
    / "historic_stock_data"
)

raw_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. CSV FILE PATH
# ============================================================

csv_file = (
    raw_folder
    / "historic_stock_data.csv"
)


# ============================================================
# 6. APPEND NEW DATA TO OLD DATA
# ============================================================

if csv_file.exists():

    # Read existing historical data
    old_df = pd.read_csv(csv_file)

    # Combine old + today's data
    historic_df = pd.concat(
        [old_df, df2],
        ignore_index=True
    )

else:

    # First time creating the file
    historic_df = df2


# ============================================================
# 7. SAVE HISTORICAL DATA
# ============================================================

historic_df.to_csv(
    csv_file,
    index=False
)


# ============================================================
# 8. INFORMATION
# ============================================================

print("Historical data saved successfully!")
print("File:", csv_file)
print("Shape:", historic_df.shape)
print("Today's date:", today)

Historical data saved successfully!
File: e:\nepse-market-report\data\raw\historic_stock_data\historic_stock_data.csv
Shape: (1388, 27)
Today's date: 2026-08-18
